# K-Means and K-Medoids (PAM)
Accompanies the notes on **K-Means and Partitioning Around Medoids**.

## 1. Lloyd's algorithm from scratch on seven points

In [ ]:
import numpy as np

def kmeans_scratch(X, centroids, n_iter=10):
    for it in range(n_iter):
        d = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)   # squared distance to each centroid
        labels = d.argmin(axis=1)                                         # assignment step
        inertia = d[np.arange(len(X)), labels].sum()
        print(f"iteration {it + 1}: labels {labels}, inertia {inertia:.2f}")
        new = np.array([X[labels == k].mean(axis=0) for k in range(len(centroids))])   # update step
        if np.allclose(new, centroids):
            break
        centroids = new
        print("   new centroids:", centroids.round(2).tolist())
    return labels, centroids

X = np.array([[1, 1], [1.5, 2], [3, 4], [5, 7], [3.5, 5], [4.5, 5], [3.5, 4.5]])
labels, centroids = kmeans_scratch(X, centroids=X[[0, 1]].astype(float))

## 2. Random starts vs k-means++

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

blobs, _ = make_blobs(n_samples=500, centers=6, cluster_std=0.6, random_state=3)
for seed in range(6):
    km = KMeans(n_clusters=6, init="random", n_init=1, random_state=seed).fit(blobs)
    print(f"random init, seed {seed}: inertia {km.inertia_:8.1f}")
km = KMeans(n_clusters=6, init="k-means++", n_init=10, random_state=0).fit(blobs)
print(f"k-means++ with n_init=10: inertia {km.inertia_:8.1f}")

## 3. PAM from scratch and the outlier experiment

In [ ]:
import numpy as np

class FirstPrinciplesKMedoids:
    """
    K-Medoids implementation using the classic PAM (Partitioning Around Medoids)
    BUILD and SWAP algorithm on pairwise distance matrices.
    """
    def __init__(self, n_clusters=3, max_iter=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.medoid_indices_ = None
        self.labels_ = None
        self.inertia_ = None

    def fit(self, X):
        n_samples = X.shape[0]
        K = self.n_clusters
        
        # 1. Compute full pairwise Euclidean distance matrix D
        # D[i, j] = ||x_i - x_j||_2
        diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
        D = np.linalg.norm(diff, axis=-1)
        
        # 2. BUILD Phase: Greedy medoid initialization
        medoids = []
        # First medoid: observation with smallest total distance to all other points
        first_medoid = np.argmin(np.sum(D, axis=1))
        medoids.append(first_medoid)
        
        # Select remaining K - 1 medoids
        for _ in range(1, K):
            # Distance from every point to its closest current medoid
            min_dist_to_medoids = np.min(D[:, medoids], axis=1)
            
            # Evaluate gain for every non-medoid candidate
            best_gain = -1
            best_candidate = -1
            for candidate in range(n_samples):
                if candidate in medoids:
                    continue
                # Gain = sum of distance reductions
                gain = np.sum(np.maximum(0, min_dist_to_medoids - D[:, candidate]))
                if gain > best_gain:
                    best_gain = gain
                    best_candidate = candidate
            medoids.append(best_candidate)
            
        medoids = np.array(medoids)
        
        # 3. SWAP Phase: Iterative improvement
        for iteration in range(self.max_iter):
            best_swap_cost_diff = 0
            best_swap_pair = None
            
            # Current total cost
            current_total_cost = np.sum(np.min(D[:, medoids], axis=1))
            
            # Test all medoid (m) and non-medoid (h) pairs
            for m_idx in range(K):
                current_m = medoids[m_idx]
                for h in range(n_samples):
                    if h in medoids:
                        continue
                    # Tentative new medoids
                    tentative_medoids = medoids.copy()
                    tentative_medoids[m_idx] = h
                    
                    # Compute new total cost
                    new_cost = np.sum(np.min(D[:, tentative_medoids], axis=1))
                    cost_diff = new_cost - current_total_cost
                    
                    if cost_diff < best_swap_cost_diff:
                        best_swap_cost_diff = cost_diff
                        best_swap_pair = (m_idx, h)
                        
            # Execute best swap if cost is reduced
            if best_swap_pair is not None and best_swap_cost_diff < -1e-6:
                m_idx, h = best_swap_pair
                medoids[m_idx] = h
            else:
                # No swap reduces cost; converged!
                break
                
        self.medoid_indices_ = medoids
        # Assign final cluster labels
        self.labels_ = np.argmin(D[:, medoids], axis=1)
        self.inertia_ = np.sum(np.min(D[:, medoids], axis=1))
        self.cluster_centers_ = X[medoids]
        return self

# a single cluster around (0, 0), plus one extreme outlier
from sklearn.cluster import KMeans
rng = np.random.default_rng(42)
X_clean = rng.normal(loc=0.0, scale=1.0, size=(100, 2))
X_contaminated = np.vstack([X_clean, [100.0, 100.0]])

kmeans = KMeans(n_clusters=1, n_init=10, random_state=42).fit(X_contaminated)
kmedoids = FirstPrinciplesKMedoids(n_clusters=1).fit(X_contaminated)
print("centre of the 100 clean points:", X_clean.mean(axis=0).round(2))
print("K-means centroid:              ", kmeans.cluster_centers_[0].round(2))
print("K-medoids medoid:              ", kmedoids.cluster_centers_[0].round(2))

## 4. Bank customers: plain standardisation vs log transform

In [ ]:
import pandas as pd
from sklearn.metrics import silhouette_score

bd = pd.read_csv("../../data/bd_train.csv")
cols = ["Average.Credit.Card.Transaction", "Balance.Transfer", "Term.Deposit", "Life.Insurance",
        "Medical.Insurance", "Average.A.C.Balance", "Personal.Loan", "Investment.in.Mutual.Fund",
        "Investment.Tax.Saving.Bond", "Home.Loan", "Online.Purchase.Amount", "Investment.in.Commudity",
        "Investment.in.Equity", "Investment.in.Derivative", "Portfolio.Balance"]
print("skewness of the columns (0 = symmetric):", bd[cols].skew().round(1).min(), "to", bd[cols].skew().round(1).max())
std = lambda d: (d - d.mean()) / d.std(ddof=0)
versions = {"standardised": std(bd[cols]),
            "log1p, then standardised": std(np.log1p(bd[cols].clip(lower=0)))}
for name, xs in versions.items():
    print(name)
    for k in range(2, 8):
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(xs)
        sil = silhouette_score(xs, km.labels_, sample_size=3000, random_state=0)
        sizes = sorted(np.bincount(km.labels_).tolist(), reverse=True)
        print(f"  K={k}: inertia {km.inertia_:9.0f}  silhouette {sil:.3f}  sizes {sizes}")

Elbow and silhouette curves for both versions:

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, xs_ in versions.items():
    inert, sils = [], []
    for k in range(2, 8):
        km_ = KMeans(n_clusters=k, n_init=10, random_state=0).fit(xs_)
        inert.append(km_.inertia_); sils.append(silhouette_score(xs_, km_.labels_, sample_size=3000, random_state=0))
    axes[0].plot(range(2, 8), inert, "o-", label=name); axes[1].plot(range(2, 8), sils, "o-", label=name)
axes[0].set_title("inertia"); axes[1].set_title("silhouette"); axes[0].set_xlabel("K"); axes[1].set_xlabel("K"); axes[1].legend()
plt.show()

## 5. Profiles of the 4 clusters (log version)

In [ ]:
xs = versions["log1p, then standardised"]
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(xs)
bd["cluster"] = km.labels_
profile = bd.groupby("cluster")[cols[:6] + ["Investment.in.Equity", "Portfolio.Balance"]].median().round(0)
profile["size"] = bd["cluster"].value_counts().sort_index()
profile["Revenue.Grid=1 (%)"] = (bd.groupby("cluster")["Revenue.Grid"].apply(lambda s: (s == 1).mean()) * 100).round(1)
print(profile.T)

## 6. K-medoids vs K-means on 300 customers

In [ ]:
sample = xs.sample(300, random_state=0).values
pam = FirstPrinciplesKMedoids(n_clusters=4).fit(sample)
km300 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(sample)
print("PAM medoid rows:", pam.medoid_indices_, " sizes:", sorted(np.bincount(pam.labels_).tolist(), reverse=True))
print("K-means sizes:  ", sorted(np.bincount(km300.labels_).tolist(), reverse=True))
from sklearn.metrics import adjusted_rand_score
print("agreement (adjusted Rand index):", round(adjusted_rand_score(pam.labels_, km300.labels_), 3))

## Try it yourself
1. Section 1 with starting centroids `X[[0, 3]]`.
2. Section 2 with `n_init=10`.
3. Profile the K = 5 clusters of the log version.